# 04 — Extract policy PDFs to a Delta table

This notebook reads the 30 policy PDFs from
`/Volumes/${catalog}/pawshield/bootstrap/policy_documents/` and lands
the extracted text in `${catalog}.pawshield.policy_documents` — the
intermediate table that the chunking notebook c0501 chunks and the index builder c0801 embeds.

**Extraction package**: `pypdfium2`. Because the policy PDFs are born-digital (no OCR needed), it is the lowest-friction choice for this corpus.

**Why a separate landing table** (rather than chunking in one pass):
keeping extraction and chunking as separate Delta tables lets each
notebook own one concern. The chunking notebook c0501 can iterate chunking strategy without
re-running PDF extraction.

**Prerequisites**: the setup notebook c0001 completed successfully — the
`bootstrap` Volume holds the 30 PDFs under `policy_documents/`.

In [0]:
dbutils.widgets.text("catalog", "genaicert")
catalog = dbutils.widgets.get("catalog")
print(f"catalog: {catalog}")

catalog: genaicert


## 1. Confirm the source PDFs are present

`read_files()` in `binaryFile` mode enumerates Volume contents with
file metadata, no Python listing needed.

In [0]:
%sql
SELECT
  path,
  length AS bytes,
  modificationTime
FROM read_files(
  '/Volumes/' || :catalog || '/pawshield/bootstrap/policy_documents/',
  format => 'binaryFile'
)
ORDER BY path
LIMIT 5;

path,bytes,modificationTime
dbfs:/Volumes/genaicert/pawshield/bootstrap/policy_documents/pp_basic_ca_v3.pdf,80995,2026-06-02T07:34:50.000Z
dbfs:/Volumes/genaicert/pawshield/bootstrap/policy_documents/pp_basic_co_v3.pdf,81828,2026-06-02T07:34:50.000Z
dbfs:/Volumes/genaicert/pawshield/bootstrap/policy_documents/pp_basic_fl_v3.pdf,81838,2026-06-02T07:34:50.000Z
dbfs:/Volumes/genaicert/pawshield/bootstrap/policy_documents/pp_basic_ga_v3.pdf,81662,2026-06-02T07:34:50.000Z
dbfs:/Volumes/genaicert/pawshield/bootstrap/policy_documents/pp_basic_il_v3.pdf,81558,2026-06-02T07:34:50.000Z


In [0]:
%sql
SELECT count(*) AS pdf_count
FROM read_files(
  '/Volumes/' || :catalog || '/pawshield/bootstrap/policy_documents/',
  format => 'binaryFile'
);

pdf_count
30


## 2. Install pypdfium2 in this notebook session

The Databricks Runtime ships with a default Python environment that
does not include pypdfium2. `%pip install` adds it for this notebook
session and triggers a Python restart.

In [0]:
%pip install --quiet pypdfium2==4.30.0

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# Widget values must be re-read after the Python restart.
catalog = dbutils.widgets.get("catalog")
print(f"catalog: {catalog}")

catalog: genaicert


## 3. Extraction UDF

The PDFs land in the cluster as binary content via Spark's `binaryFile`
source. A `pandas_udf` runs pypdfium2 on each row's bytes and returns
the concatenated page text. Page-level granularity is unnecessary
for policy contracts; Section-level granularity is the right unit and
is handled by the structure-aware chunker in c0501.

In [0]:
from datetime import datetime, timezone
import pandas as pd
import pypdfium2 as pdfium
from pyspark.sql.functions import (col, current_timestamp, input_file_name,
                                   lit, pandas_udf, regexp_extract)
from pyspark.sql.types import StringType


@pandas_udf(StringType())
def extract_pdf_text(content: pd.Series) -> pd.Series:
    """Extract full text from PDF bytes using pypdfium2.

    Returns the concatenated text of every page, with a single newline
    separating pages. Empty string if extraction yields no text (which
    indicates a scanned-image PDF — out of scope for the born-digital
    policy corpus)."""
    out = []
    for blob in content:
        if blob is None:
            out.append("")
            continue
        pdf = pdfium.PdfDocument(blob)
        try:
            pages = []
            for i in range(len(pdf)):
                page = pdf[i]
                textpage = page.get_textpage()
                pages.append(textpage.get_text_range())
                textpage.close()
                page.close()
            out.append("\n".join(pages))
        finally:
            pdf.close()
    return pd.Series(out)

## 4. Read PDFs and parse the filename

Each filename follows `<tier-prefix>_<state>_v3.pdf` (per the
inventory). The prefix maps to a tier:

* `pp_basic`  → WhiskerBasic
* `pp_plus`   → PawPlus
* `pp_premier` → FureverPremier

Parsing with regex keeps the metadata derivation visible in the
code, rather than depending on a Python lookup table buried in a UDF.

In [0]:
src = f"/Volumes/{catalog}/pawshield/bootstrap/policy_documents/"

raw = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(src)
    .withColumn("source_path", col("path"))
    .withColumn(
        "doc_id",
        regexp_extract("path", r"/([^/]+)\.pdf$", 1),
    )
    .withColumn(
        "tier_prefix",
        regexp_extract("doc_id", r"^(pp_basic|pp_plus|pp_premier)_", 1),
    )
    .withColumn(
        "state",
        regexp_extract("doc_id", r"_([a-z]{2})_v3$", 1),
    )
)
print(f"src: {src}")

src: /Volumes/genaicert/pawshield/bootstrap/policy_documents/


In [0]:
display(raw.select("doc_id", "tier_prefix", "state", "length"))

doc_id,tier_prefix,state,length
pp_basic_or_v3,pp_basic,or,83720
pp_plus_ca_v3,pp_plus,ca,82098
pp_premier_ma_v3,pp_premier,ma,81995
pp_basic_fl_v3,pp_basic,fl,81838
pp_basic_co_v3,pp_basic,co,81828
pp_basic_ga_v3,pp_basic,ga,81662
pp_plus_ga_v3,pp_plus,ga,81610
pp_premier_il_v3,pp_premier,il,81600
pp_basic_il_v3,pp_basic,il,81558
pp_premier_ny_v3,pp_premier,ny,81555


In [0]:
from pyspark.sql.functions import upper, when

extracted = (
    raw.withColumn("text", extract_pdf_text(col("content")))
    .withColumn("state", upper(col("state")))
    .withColumn(
        "tier",
        when(col("tier_prefix") == "pp_basic", lit("WhiskerBasic"))
        .when(col("tier_prefix") == "pp_plus", lit("PawPlus"))
        .when(col("tier_prefix") == "pp_premier", lit("FureverPremier")),
    )
    .withColumn("extracted_at", current_timestamp())
    .select("doc_id", "tier", "state", "source_path", "extracted_at", "text")
)

## 5. Sanity-check Sarah Chen's policy

Sarah lives in TX and holds the PawPlus tier. The Section 4.7 string
("Chronic condition exclusions and reimbursement handling") must
appear in `pp_plus_tx_v3.pdf` — this is the canonical "right chunk"
the rest of the book retrieves.

In [0]:
from pyspark.sql.functions import length as len_, instr

check = extracted.filter(col("doc_id") == "pp_plus_tx_v3").select(
    "doc_id", "tier", "state",
    len_("text").alias("char_count"),
    instr("text", "Chronic condition exclusions").alias("section_4_7_offset"),
)

In [0]:
display(check)

doc_id,tier,state,char_count,section_4_7_offset
pp_plus_tx_v3,PawPlus,TX,101104,9924


## 6. Write to Delta

One row per PDF. Overwrite mode makes this notebook idempotent
across re-runs (extraction is deterministic on the same source PDFs).

In [0]:
target = f"{catalog}.pawshield.policy_documents"
print(f"target: {target}")

target: genaicert.pawshield.policy_documents


In [0]:
(
    extracted.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target)
)

print(f"Wrote {extracted.count()} rows to {target}")

Wrote 30 rows to genaicert.pawshield.policy_documents


In [0]:
%sql
SELECT
  tier,
  state,
  length(text) AS chars
FROM IDENTIFIER(:catalog || '.pawshield.policy_documents')
ORDER BY tier, state;

tier,state,chars
FureverPremier,CA,98011
FureverPremier,CO,101188
FureverPremier,FL,101194
FureverPremier,GA,100843
FureverPremier,IL,101266
FureverPremier,MA,101692
FureverPremier,NY,101313
FureverPremier,OR,99589
FureverPremier,TX,98715
FureverPremier,WA,98795


## Adjacent ingest-layer primitives (not used here)

This notebook scopes to `pypdfium2` + `binaryFile` because the
PolicyPal corpus is born-digital. Three other Databricks-native
primitives would slot in for related
ingest scenarios — none demonstrated here, but each named so the
reader knows where in the docs to look:

* **Auto Loader (`cloudFiles`)** — incremental file ingest for
  streaming/append-style sources. When new policy PDFs arrive
  weekly, `cloudFiles` schedules the parse on a trigger instead of
  batching against a static directory. Source:
  docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader.
* **`ai_parse_document`** — a SQL function that parses PDFs (text
  AND tables AND images) inside the warehouse. Use when the PDFs
  are mixed-content (vet invoices with line-item tables, scanned
  forms) and a one-step SQL parse beats orchestrating an OCR
  fallback. Caps: 100MB / 500 pages per call. Source:
  docs.databricks.com/aws/en/sql/language-manual/functions/ai_parse_document.
* **`ai_mask`** — source-side PII masking applied at parse time
  so PII never lands in Delta or the Vector Search index. The
  right primitive for "the index must never contain SSNs" rules;
  contrast with runtime AI Gateway PII Redaction (applied at serving time, not parse time). Source:
  docs.databricks.com/aws/en/sql/language-manual/functions/ai_mask.

## What's next

* The chunking notebook c0501 reads `policy_documents` and produces `policy_chunks`, splitting
  on section headings so Section 4.7 of `pp_plus_tx_v3` stays whole.
* The index builder c0801 creates a Vector Search Delta Sync index over `policy_chunks`
  with `state` and `tier` as filterable metadata columns.